## Clinical Data Engineering: Relational Schema Mapping & SQL Load

<div style="padding: 15px; border: 1px solid #003152; border-radius: 5px; background-color: #fff5ef; color: #003152;">

**Project:** Quantifying Clinical Gaps in Valvular Heart Disease: Using the Medication Burden Index (MBI) to Triage High-Complexity Intervention Candidates.  
**Source:** Cleaned Project Health Leon Dataset ($N=152$)  
**Structure:** Snowflake Schema with Many-to-Many Bridge Tables  
**Version:** 1.1 (Post-Audit)  

---
**Technical Focus:** This notebook transitions from a flat clinical file to a relational database model. It implements an Object-Oriented Normalizer to handle complex clinical relationships (polypharmacy and longitudinal diagnoses).  
**Key Goal:** Ensure 3NF (Third Normal Form) compatibility for integration into a MySQL production environment.

</div>

### 🪢 Relational Schema Design & Referential Integrity
---

This notebook serves as the Load (L) in our ETL pipeline. The transition from a 2D dataframe to a Snowflake schema is essential for maintaining a high-integrity clinical database. We implement a schema that enforces Primary and Foreign Key relationships, ensuring that every medication dose and diagnosis is traceable to a specific patient event.

In [31]:
# 1. IMPORTS
# ==========================================

import pandas as pd
import numpy as np
from IPython.display import display, Markdown, HTML
import re
import missingno as msno
import matplotlib.colors as mcolors
import statsmodels.api as sm
from sklearn.metrics import roc_curve, auc, f1_score, precision_score, recall_score, confusion_matrix
from scipy.stats import kruskal, chi2_contingency
import scipy.stats as sstats
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import mysql.connector
from mysql.connector import Error
import sys
import os
from pathlib import Path

# File path for loading the 'Medical Brain' source
sys.path.append(str(Path.cwd().parent))
from config import ClinicalConfig 

### ⚙️ Class-Based normalization strategy
---
The `Normalizer` class ensures that varied clinical terms are mapped to a uniform `dim_diagnoses` table. This prevents the "Double Counting" of pathologies due to spelling variations and allows for consistent categorization across the entire cardiology cohort:

* **Self-Documenting Code:** Every method corresponds to a specific table in our SQL schema (e.g., `get_dim_medications` $\rightarrow$ `dim_medications`).

* **Data Integrity:** Dropping redundant string columns in favor of integer keys ensures we follow DRY (Don't Repeat Yourself) principles and reduces storage footprint.

* **Schema Consistency:** Ensuring that `med_id` and `dx_id` assignments remain consistent across different data batches.  

In [32]:
# 2. NORMALIZER CLASS
# ==========================================

class Normalizer:
    """
    A centralized hub for mapping raw clinical observations into a relational Snowflake schema.
    """
    def __init__(self, dataframe, config):
        self.df = dataframe
        self.config = config

    def get_dim_diagnoses(self):
        '''
        Create normalized dimension dataframe for diagnoses 
        '''
        # Concatenate anatomy_pre procedure and anatomy_post procedure diagnoses into a single column to extract unique strings
        da = pd.concat([
            self.df['anatomy_pre'],
            self.df['anatomy_post']
        ]).str.split(',').explode().str.strip().dropna().unique()

        # Turn the list into a normalized dataframe with sequential ids
        da = pd.DataFrame(da, columns=['diagnosis'])
        da.insert(0, 'dx_id', range(1, len(da) + 1))
        return da
    
    def get_dim_medications(self):
        '''
        Create normalized dimension dataframe for medications 
        '''
        # Initialize an empty list to store medication rows
        rows = []

        # Outer Loop: Go through each class (key) and its list of drugs (value)
        for med_class, drugs in self.config.medication_classes.items():
            
            # Inner Loop: Go through every drug inside that specific class list
            for drug_name in drugs:
                
                # Create a dictionary per medication row
                row = {
                    'med_name': drug_name,
                    'med_class': med_class
                }
                
                # Add the row to our master list
                rows.append(row)

        # Turn the list into a normalized dataframe with sequential ids
        da = pd.DataFrame(rows)
        da.insert(0, 'med_id', range(1, len(da) + 1))
        return da
    
    def get_bridge_medications(self, dim_table):
        '''
        Create bridge dataframe for medications and patients
        '''
        # Initialize an empty list to store medication rows
        bridge_med_list = []

        # Loop through each medication category defined in ClinicalConfig
        for med_class_key in self.config.medication_classes.keys():
            # Construct the column names used in the cleaned df_output.csv
            prefix = med_class_key.lower()
            name_col = f"{prefix}_name"
            tdd_col = f"{prefix}_tdd"
            
            if name_col in self.df.columns:
                # Extract patient_id, the drug name, and the dose
                temp = self.df[['patient_id', name_col, tdd_col]].copy()
                
                # Filter out 'None' or empty prescriptions
                temp = temp[temp[name_col] != 'None'].dropna(subset=[name_col])
                temp.columns = ['patient_id', 'med_name', 'tdd']
                bridge_med_list.append(temp)

        # Combine all categories into one "long" dataframe
        da = pd.concat(bridge_med_list)

        # Map the drug name to its med_id using the dim_medications dataframe
        da = da.merge(dim_table, on='med_name')[['patient_id', 'med_id', 'tdd']].sort_values(by='patient_id')
        return da
    
    def _normalize_anatomy(self, col_name, timing_label):
        '''
        Method for handling diagnoses strings into multiple rows
        '''
        temp = self.df[['patient_id', col_name]].copy()
        
        # Split string by comma and explode into multiple rows
        temp[col_name] = temp[col_name].str.split(',')
        temp = temp.explode(col_name).dropna(subset=[col_name])
        
        # Clean whitespace
        temp[col_name] = temp[col_name].str.strip()
        temp.columns = ['patient_id', 'diagnosis']
        
        # Add the timing indicator column
        temp['timing'] = timing_label
        return temp
    
    def get_bridge_diagnoses(self, dim_table):
        '''
        Create bridge dataframe for medications and diagnoses
        '''
        # Generate the two raw sets with their labels
        bridge_pre = self._normalize_anatomy('anatomy_pre', 'Pre')
        bridge_post = self._normalize_anatomy('anatomy_post', 'Post')

        # Concatenate them into a single column
        da = pd.concat([bridge_pre, bridge_post])

        # Map to your dim_diagnoses IDs and keep the timing column
        da = da.merge(dim_table, on='diagnosis')

        # Add clinical indicators
        clinical_indicators = [
            'patient_id', 'approach', 'sev_ms','sev_mr','sev_as','sev_ar','sev_ph','sev_septal','la_thrombus','la_mass','mixed_mitral','mixed_aortic','multi_valve','ph_significant','has_shunt',
            'afib_present','complexity_score'
        ]
        da = da.merge(self.df[clinical_indicators], on='patient_id', how='left')
        
        # Drop the column diagnosis since there is dx_id
        da = da.drop(columns=['diagnosis'])
        
        # Reorder columns for the final table
        final_cols = ['patient_id', 'dx_id', 'timing'] + [c for c in clinical_indicators if c != 'patient_id']
        return da[final_cols].sort_values(by=['patient_id', 'timing'])

### 🛡️ Programmatic Schema Deployment & Schema Integrity
---
Moving away from flat file analysis requires strict data governance. The `DatabaseManager` class functions as the structural enforcement layer for our database, transforming data types to match clinical analysis requirements (e.g., mapping binary integer indicators into strict SQL BOOLEAN flags).

* **Constraint Mapping:** Explicitly binds `PRIMARY KEY` and `FOREIGN KEY` definitions to preserve patient-to-event relationships.

* **Batch Consistency:** Centralizing the connection and execution pipeline ensures that multi-table data remains synchronized.

* **Transactional Safety (ACID Compliance):** If any table compilation encounters an exception, the entire layout rolls back, preventing corrupt, orphan, or partially initialized database schemas.

In [33]:
# 3. DATABASEMANAGER CLASS
# ==========================================

# Database credentials
db_config = {
    'user': 'root',
    'password': 'pass',
    'database': 'cardio_mbi',
}

class DatabaseManager:
    """
    Manage the connection, DDL schema generation, and transactional loading 
    of the normalized clinical snowflake entities into the target relational instance.
    """
    def __init__(self, config: dict):
        self.config = config
        self.conn = None
        self.cursor = None

    def connect(self):
        """
        Establish connection to the database and initializes an active operational cursor.
        """
        try:
            self.conn = mysql.connector.connect(**self.config)
            self.cursor = self.conn.cursor()

            # Execution visibility for verification
            self.cursor.execute("USE cardio_mbi")
            print(f"Successfully connected. Relational instance in use: '{self.conn.database}'")
        except Error as e:
            # Return error without a traceback
            print(f"CRITICAL ERROR: Failed to establish SQL connection: {e}")
        
    def create_schema(self):
        """
        Execute explicit DDL operations to establish structural and referential integrity rules 
        (Primary Keys, Foreign Keys).
        """

        # Defensive check before attempting execution
        if not self.cursor:
            print("Database connection must be active before creating tables.")

        # Dictionary of table definitions structured in order of referential dependency
        queries = {
            "dim_diagnoses": """
                CREATE TABLE IF NOT EXISTS dim_diagnoses (
                    dx_id INT PRIMARY KEY,
                    diagnosis VARCHAR(255) UNIQUE NOT NULL
                );
            """,
            "dim_medications": """
                CREATE TABLE IF NOT EXISTS dim_medications (
                    med_id INT PRIMARY KEY,
                    med_name VARCHAR(255) UNIQUE NOT NULL,
                    med_class VARCHAR(255) NOT NULL
                );
            """,
            "fact_patient": """
                CREATE TABLE IF NOT EXISTS fact_patient (
                    patient_id INT PRIMARY KEY,
                    age FLOAT,
                    gender ENUM('Male', 'Female'),
                    sbp_r FLOAT,
                    dbp_r FLOAT,
                    heart_rate FLOAT,
                    osat FLOAT,
                    weight FLOAT,
                    la_size_score FLOAT NOT NULL,
                    lvef_score INT NOT NULL,
                    complexity_score INT NOT NULL,
                    mbi FLOAT NOT NULL,
                    ms_mg_mmhg_imputed FLOAT NOT NULL,
                    ao_v2_max_imputed FLOAT NOT NULL,
                    rvsp_imputed FLOAT NOT NULL,
                    lvidd_imputed FLOAT NOT NULL
                );
            """,
            "bridge_medications": """ 
                CREATE TABLE IF NOT EXISTS bridge_medications (
                    patient_id INT,
                    med_id INT,
                    tdd FLOAT,
                    PRIMARY KEY (patient_id, med_id),
                    FOREIGN KEY (patient_id) REFERENCES fact_patient(patient_id),
                    FOREIGN KEY (med_id) REFERENCES dim_medications(med_id)
                );
            """,
            "bridge_diagnoses": """
                CREATE TABLE IF NOT EXISTS bridge_diagnoses (
                    patient_id INT,
                    dx_id INT,
                    timing ENUM('Pre', 'Post'),
                    approach ENUM ('Native', 'Percutaneous', 'Surgical') ,
                    sev_ms FLOAT,
                    sev_mr FLOAT,
                    sev_as FLOAT,
                    sev_ar FLOAT,
                    sev_ph FLOAT,
                    sev_septal INT,
                    la_thrombus BOOLEAN DEFAULT 0,
                    la_mass BOOLEAN DEFAULT 0,
                    mixed_mitral BOOLEAN DEFAULT 0,
                    mixed_aortic BOOLEAN DEFAULT 0,
                    multi_valve BOOLEAN DEFAULT 0,
                    ph_significant BOOLEAN DEFAULT 0,
                    has_shunt BOOLEAN DEFAULT 0,
                    afib_present BOOLEAN DEFAULT 0,
                    complexity_score INT,
                    PRIMARY KEY (patient_id, dx_id, timing),
                    FOREIGN KEY (patient_id) REFERENCES fact_patient(patient_id),
                    FOREIGN KEY (dx_id) REFERENCES dim_diagnoses(dx_id)
                );
            """
        }

        try:
            # Start transaction to enable rollback if any query breaks
            self.conn.start_transaction()
            for table_name, query in queries.items():
                self.cursor.execute(query)

            self.conn.commit()
            # Execution visibility for verification
            print("Successfully created database tables.")
        except Error as e:
            # Return error without a traceback
            print(f"CRITICAL ERROR: Failed to execute DDL: {e}")
            self.conn.rollback()
    
    def populate_tables(self, dim_diagnoses_df, dim_medications_df, fact_df, bridge_meds_df, bridge_dx_df):
        """
        Executes safe relational bulk insertions into MySQL tables and 
        referential integrity by populating Dimensions first, 
        followed by Facts, and lastly Associative Bridge entities.
        """

        # Defensive check before attempting execution
        if not self.cursor:
            print("Database connection must be active before creating tables.")

        try:
            self.conn.start_transaction()
            print("Initiating transaction: Populating database...")

            # Populate dimensions
            # dim_diagnoses
            query_dim_dx = """
                INSERT INTO dim_diagnoses (dx_id, diagnosis)
                VALUES (%s, %s);
            """
            # Convert NaN to None so MySQL registers them as NULL, and pass rows as a list of tuples
            val_dim_dx = [tuple(x) for x in dim_diagnoses_df.replace({np.nan: None}).values]
            self.cursor.executemany(query_dim_dx, val_dim_dx)
            print(f"Successfully loaded {self.cursor.rowcount} records into dim_diagnoses.")

            # dim_medications
            query_dim_meds = """
                INSERT INTO dim_medications (med_id, med_name, med_class)
                VALUES (%s, %s, %s);
            """
            val_dim_meds = [tuple(x) for x in dim_medications_df.replace({np.nan:None}).values]
            self.cursor.executemany(query_dim_meds, val_dim_meds)
            print(f"Successfully loaded {self.cursor.rowcount} records into dim_medications.")

            # fact_patients
            query_fact = """
                INSERT INTO fact_patient (
                patient_id, age, gender, sbp_r, dbp_r, heart_rate, osat, weight, la_size_score,
                lvef_score, complexity_score, mbi, ms_mg_mmhg_imputed, ao_v2_max_imputed,
                rvsp_imputed, lvidd_imputed
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            val_fact = [tuple(x) for x in fact_df.replace({np.nan: None}).values]
            self.cursor.executemany(query_fact, val_fact)
            print(f"Successfully loaded {self.cursor.rowcount} records into fact_patient.")

            # bridge medications
            query_bridge_meds = """
                INSERT INTO bridge_medications (patient_id, med_id, tdd)
                VALUES (%s, %s, %s)
            """
            val_bridge_meds = [tuple(x) for x in bridge_meds_df.replace({np.nan: None}).values]
            self.cursor.executemany(query_bridge_meds, val_bridge_meds)
            print(f"Successfully loaded {self.cursor.rowcount} records into bridge_medications.")

            # bridge diagnoses
            query_bridge_dx = """
                INSERT INTO bridge_diagnoses (
                patient_id, dx_id, timing, approach, sev_ms, sev_mr, sev_as, sev_ar, sev_ph, 
                sev_septal, la_thrombus, la_mass, mixed_mitral, mixed_aortic, multi_valve, ph_significant, has_shunt, afib_present, complexity_score
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            val_bridge_dx = [tuple(x) for x in bridge_dx_df.replace({np.nan: None}).values]
            self.cursor.executemany(query_bridge_dx, val_bridge_dx)
            print(f"Successfully loaded {self.cursor.rowcount} records into bridge_diagnoses.")

            self.conn.commit()
            print("\nTransaction committed successfully. Database layer is fully populated")
        except Error as e:
            # Return error without a traceback
            print(f"CRITICAL ERROR: Failed to populate existing SQL tables: {e}")
            self.conn.rollback()        

    def drop_tables (self):
        """
        Safely drops all relational table layers in reverse order of 
        their dependencies to prevent Foreign Key constraint violations.
        """

        # Defensive check before attempting execution
        if not self.cursor:
            print("Database connection must be active before dropping tables.")
        
        tables_to_drop = [
            "bridge_diagnoses",
            "bridge_medications",
            "fact_patient",
            "dim_diagnoses",
            "dim_medications"
        ]

        try:
            self.conn.start_transaction()
            for table_name in tables_to_drop:
                query = f"DROP TABLE IF EXISTS {table_name};"
                self.cursor.execute(query)

            self.conn.commit()
            # Execution visibility for verification
            print("Database slate wiped clean.")
        except Error as e:
            # Return error without a traceback
            print(f"CRITICAL ERROR: Failed to execute DDL: {e}")
            self.conn.rollback()

    def truncate_tables(self):
        """Wipes table data and resets auto-increment keys while preserving structure."""
        # Temporarily disable foreign keys to allow data wiping without conflicts
        self.cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")
        
        tables = ['bridge_diagnoses', 'bridge_medications', 'fact_patient', 'dim_diagnoses', 'dim_medications']
        for table in tables:
            self.cursor.execute(f"TRUNCATE TABLE {table};")
            
        self.cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")

    def close(self):
        """
        Gracefully disconnects to prevent orphan connection states.
        """
        if self.cursor:
            self.cursor.close()
        if self.conn and self.conn.is_connected():
            self.conn.close()
            print("Database connection successfully terminated.")

In [34]:
# 4. TABLES AND THEIR STRUCTURE
# ==========================================
output = r'..\data\processed\df_output.csv'
if not os.path.exists(output):
    print(f"CRITICAL ERROR: File not found at {os.path.abspath(output)}")
else:
    load = pd.read_csv(output, dtype={'ID': str})
    print(f"Successfully loaded {len(load)} patient records.")

Successfully loaded 152 patient records.


In [35]:
# Type handling before table creation
float_cols = [
    'age', 'sbp_r', 'dbp_r', 'heart_rate', 'sev_ms', 'sev_mr', 'sev_as', 'sev_ar', 'sev_ph', 'sev_septal'
]

# str_cols = [
#     'patient_id'
# ]

binary_cols = [
    'la_thrombus', 'la_mass', 'mixed_mitral', 'mixed_aortic', 'multi_valve', 'ph_present', 'ph_significant', 
    'has_shunt', 'afib_present'
]

int_cols = [
    'sev_septal'
]

load[float_cols] = load[float_cols].astype(float)
# load[str_cols] = load[str_cols].astype(str)
load[binary_cols] = load[binary_cols].astype(bool)
load[int_cols] = load[int_cols].round(0).astype(int)

df = load.copy()


### ⛓️ Resolving Many-to-Many Clinical Relationships
---

Clinical reality is rarely one-to-one. A single patient often presents with multiple valvular pathologies (e.g., Mixed Mitral Disease) and complex medication regimens.

**Bridge Table Implementation:**

* `bridge_medications:` Unpivots the flat medication columns to link `patient_id` to `med_id`, preserving the **Total Daily Dose (TDD)** as a relationship attribute.

* `bridge_diagnoses:` Maintains a longitudinal record by separating Pre and Post procedure conditions while mapping them to a centralized `dim_diagnoses` catalog.

### 💡 Hybrid Third Normal Form (3NF) Strategy
---
While we maintain 3NF for our Dimension and Fact tables, we utilize Denormalization for clinical attributes in our Bridge diagnoses table. By including `complexity_score`, `severity` indicators, and `timing` labels in the bridge, we optimize the structure for analytical queries.

**The Benefit:** This reduces the "Join Depth" required for common tasks. Calculating the **Average pulmonary hypertension severity per diagnosis** can be executed via a single join between `bridge_diagnoses` and `dim_diagnoses`, bypassing the heavy computational overhead of the central Fact table.

In [36]:
# Normalizer class instantiation
normalizer = Normalizer(df, ClinicalConfig())

# Fact table
fact_patient = df[['patient_id', 'age', 'gender', 'sbp_r', 'dbp_r', 'heart_rate', 'osat', 'weight', 'la_size_score', 'lvef_score', 'complexity_score', 'mbi',
 'ms_mg_mmhg_imputed', 'ao_v2_max_imputed', 'rvsp_imputed', 'lvidd_imputed']]

# Diagnoses table
dim_diagnoses = normalizer.get_dim_diagnoses()

# Medications table
dim_medications = normalizer.get_dim_medications()

In [37]:
# Bridge table for diagnoses
bridge_diagnoses = normalizer.get_bridge_diagnoses(dim_diagnoses)

# Bridge table for medications
bridge_medications = normalizer.get_bridge_medications(dim_medications)

### 📍 Clinical Data Mapping and Dictionary
---

To ensure analytical reproducibility and pipeline traceability, qualitative clinical classifications from `ClinicalConfig` are mapped to categorical identifiers, ordinal scales, or binary flags prior to relational injection:

| Column Name(s) | Modeled Type | Modeled Values / Scale | Clinical Meaning & Analytic Interpretation |
|:---|:---|:---|:---|
| **`approach`** | `VARCHAR` | `'Native'`, `'Percutaneous'`, `'Surgical'` | Specifies the structural state of the evaluated valve. |
| **`timing`** | `ENUM` | `'pre'`, `'post'` | Discriminator capturing longitudinal clinical trajectory (Pre-intervention Baseline vs. Post-intervention Follow-up). |
| **`la_size_score`** | `FLOAT` | `1.0`, `2.5`, `4.0` | Ordinal clinical indexing for Left Atrium enlargement: Mildly, Moderately, or Severely Enlarged. |
| **`lvef_score`** | `INT` | `75`, `62`, `52`, `35`, `25` | Discrete clinical strata representing: Hyperdynamic, Normal, Borderline, Moderately Reduced, and Severely Reduced ejection fractions. |
| **Valve Severity** <br>(`sev_ms`, `sev_mr`, `sev_as`, `sev_ar`, `sev_ph`) | `FLOAT` | `0.0` to `3.5` <br>*(0.5 increments)* | Fine-grained ordinal ranking: `0` (None), `1.0` (Mild), `2.0` (Moderate), `3.0` (Severe), up to `3.5` (Critical). |
| **`mixed_mitral`** | `BOOLEAN` | `1` (True), `0` (False) | Phenotypic flag indicating simultaneous stenosis and regurgitation on the mitral valve. |
| **`mixed_aortic`** | `BOOLEAN` | `1` (True), `0` (False) | Phenotypic flag indicating simultaneous stenosis and regurgitation on the aortic valve. |
| **`multi_valve`** | `BOOLEAN` | `1` (True), `0` (False) | Composite marker indicating simultaneous pathological involvement across multiple heart valves. |
| **`complexity_score`** | `INT` | `Integer` | Cumulative, additive risk index quantifying total high-risk diagnostic indicators for advanced hemodynamic alterations. |


In [38]:
# 4. DATABASE SCHEMA
# ==========================================
db_manager = DatabaseManager(db_config)
db_manager.connect()

Successfully connected. Relational instance in use: 'cardio_mbi'


### ❄️ Snowflake Schema (Reverse-Engineered via MySQL Workbench)
---

To ensure the dataset is analysis-ready for statistical computing, the database abstracts messy clinical realities into a relational schema. Longitudinal timing variables and intervention complexities are maintained as relationship attributes within the bridge dimensions, allowing for fast, targeted query extraction without risking the duplicate counting of core patient parameters.

![Database schema](../assets/database_schema.png)

In [39]:
db_manager.create_schema()

Successfully created database tables.


In [40]:
db_manager.populate_tables(
    dim_diagnoses_df=dim_diagnoses,
    dim_medications_df=dim_medications,
    fact_df=fact_patient,
    bridge_meds_df=bridge_medications,
    bridge_dx_df=bridge_diagnoses
)

Initiating transaction: Populating database...
Successfully loaded 71 records into dim_diagnoses.
Successfully loaded 36 records into dim_medications.


Successfully loaded 152 records into fact_patient.
Successfully loaded 470 records into bridge_medications.
Successfully loaded 314 records into bridge_diagnoses.

Transaction committed successfully. Database layer is fully populated


In [41]:
# Method for dropping all existing tables
# db_manager.drop_tables()

# Method for truncating all existing tables
# db_manager.truncate_tables()

In [42]:
db_manager.close()

Database connection successfully terminated.
